# Full monthly comparison: realized merger payoffs only

The original Paper15 RF/XGBoost five/ten-year models, forecasts, RET scores,
training labels and selected baskets remain frozen. This zero-fit replay adds
verified realized cash/share merger entitlements for EV, CDK, XEC and XLNX.
It does not invent old-stock month-end prices, drop selected stocks or refit.

Full period: January 2020–December 2023, **48 monthly returns**. A separately
initialized February 2020–December 2023 comparator has47returns. Primary net
cost is10bp one-way; no intramonth stops;50%long/50%short. Risk metrics use
monthly returns and month-end marks; same-close entry remains idealized.

[Method and legal evidence](../docs/MONTHLY_MERGER_REPLAY_20261026.md).
This notebook reads independently verified small reports only. **RunAll never
loads models, fits, rebuilds labels/RET, reads source quotes or replays trades.**
The strict monthly benchmark and fourteen-month report remain preserved.


In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
local = ROOT / "configs/monthly_merger_replay.local.json"
private = json.loads(local.read_text()) if local.exists() else {}
MERGER_REPLAY_RUN = Path(private.get("output_directory", ROOT / "outputs/paper15_monthly_merger_replay/20261026-v1"))
print("Workspace:", ROOT)
print("Saved replay:", MERGER_REPLAY_RUN)


In [ ]:
from dmv2.monthly_merger_reporting import read_monthly_merger_replay
merger_saved = None
try:
    merger_saved = read_monthly_merger_replay(MERGER_REPLAY_RUN)
except FileNotFoundError:
    print("Private replay evidence is unavailable. No compute started.")
else:
    print(merger_saved["status"], "new fits:", merger_saved["fits_completed"])
    if merger_saved["status"] == "COMPLETE_VERIFIED":
        print(merger_saved["full_period"])
        print("Monthly paths:", merger_saved["complete_paths"], "complete;", merger_saved["withheld_paths"], "withheld")
        print("Original forecasts, labels, RET and selected baskets unchanged.")
        display(merger_saved["primary"])
        display(merger_saved["annual"].query("period == 'full48' and cost_bps == 10").pivot(index="arm", columns="year", values="total_return"))
        display(merger_saved["comparison"])
        display(merger_saved["learning_scope_audit"])
        if "figure_path" in merger_saved:
            from IPython.display import Image
            display(Image(filename=merger_saved["figure_path"]))
    else:
        print("Independent verification pending.")
